In [ ]:
import os
import re
import json
from transformers import AutoTokenizer

In [ ]:
my_cache_dir = r"E:\Models"
BLOCK_SIZE = 1024  
OVERLAP = 128
STRIDE = BLOCK_SIZE - OVERLAP

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",cache_dir=my_cache_dir)

In [ ]:

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
def tokenize_fn(examples):
    tokens = tokenizer(examples["text"],truncation=True,padding="max_length",max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [ ]:


all_chunks = []
file_directories = []
data = []

In [ ]:
nextjs_path = "./docs/next.js-canary/docs"
for file in os.listdir(nextjs_path):
    current_file = nextjs_path + "/" + file
    if not os.path.isdir(current_file):
        if ".mdx" in nextjs_path:
            file_directories.append(f"{nextjs_path}/{file}")
    else:
        for sub_files in os.listdir(f"{nextjs_path}/{file}"):
            current_file = nextjs_path + "/" + file + "/" + sub_files

            if not os.path.isdir(current_file):
                if ".mdx" in sub_files:
                    file_directories.append(current_file)
            else:
                for s_f in os.listdir(f"{nextjs_path}/{file}/{sub_files}"):

                    current_file = (
                        nextjs_path + "/" + file + "/" + sub_files + "/" + s_f
                    )
                    if ".mdx" in s_f:
                        file_directories.append(current_file)


In [ ]:
def clean_mdx_content(text):
    """Strips out messy React import headers from the top of MDX files."""
    text = re.sub(r'^import\s+.*?\s+from\s+[\'"].*?[\'"];?\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^export\s+default\s+.*$', '', text, flags=re.MULTILINE)
    return text.strip()

In [ ]:
framework = "NextJs"
for file in file_directories:
    with open(file, "r") as f:
        basename = os.path.basename(f.name)
        purpose = re.sub(r'[0-9]', '', basename).removeprefix("-").removesuffix(".mdx")
        content = clean_mdx_content(f.read())
        
        header = f"Framework: {framework} \n Purpose:{purpose}\n Content:"
        full_text = header + "\n" + content
        
        tokens = tokenizer.encode(full_text, add_special_tokens=False)
                
        for i in range(0, len(tokens), STRIDE):
            chunk_tokens = tokens[i:i + BLOCK_SIZE]
            chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
            all_chunks.append({
                "purpose": purpose,
                "text": chunk_text,
                "framework": "NextJs",
                "source": basename,
            })



In [ ]:
output_file = "chunks/nextjs.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print(f"Success! Generated {len(all_chunks)} chunks in {output_file}")

Success! Generated 494 chunks in chunks/nextjs.jsonl
